# 03 — Mean–variance optimization

**Purpose:** Solve the same problem as the API: minimize $\lambda w'\Sigma w - \mu'w$ subject to $w \ge 0$, $\sum w = 1$.

Risk profiles in the app map to $\lambda \in \{10, 5, 1\}$ (conservative → aggressive).


In [ ]:
# Allow imports from backend/ (start Jupyter from `notebooks/` or project root)
from pathlib import Path
import sys
for base in [Path.cwd(), Path.cwd() / "notebooks"]:
    for b in [base, base.parent]:
        be = b / "backend"
        if (be / "optimize.py").exists():
            sys.path.insert(0, str(be))
            break
    else:
        continue
    break
else:
    raise RuntimeError("Could not find backend/ — `cd` to smart-portfolio-system or notebooks/")


In [ ]:
from optimize import run_optimization, optimize_portfolio, fetch_price_data, compute_returns, estimate_parameters

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
START, END = "2022-01-01", "2024-12-31"

weights, mu, cov = run_optimization(TICKERS, START, END, lambda_risk=5)
weights


## Sensitivity: weights vs $\lambda$


In [ ]:
import pandas as pd
import numpy as np

lambdas = [10, 5, 1]
rows = []
for lam in lambdas:
    w, _, _ = run_optimization(TICKERS, START, END, lambda_risk=lam)
    rows.append(pd.Series(w, name=f"lambda={lam}"))
pd.DataFrame(rows).T.round(4)


In [ ]:
import matplotlib.pyplot as plt

rows = {}
for lam in lambdas:
    w, _, _ = run_optimization(TICKERS, START, END, lambda_risk=lam)
    rows[f"λ={lam}"] = [w[t] for t in TICKERS]
wmat = pd.DataFrame(rows, index=TICKERS)
wmat.plot(kind="bar", figsize=(10, 4), title="Optimized weights by risk aversion λ")
plt.ylabel("weight")
plt.tight_layout()
plt.legend(title="profile")
plt.show()


### Takeaway
Higher $\lambda$ penalizes variance more heavily; weights shift toward lower-volatility names **in-sample**, subject to estimation noise.
